# 4일차 실습 ① — SAM 으로 객체 분할

부경대학교 교내 컴퓨터비전 부트캠프 · 2026. 8. 6. · 2차시 (10:30 – 11:20)

---

박스가 아니라 **픽셀 단위 마스크**를 다룹니다. SAM 은 클래스를 모르는 대신,
가리킨 것을 오려 내는 일을 아주 잘합니다.

| STEP | 하는 일 |
|---|---|
| 0 | 환경 준비 |
| 1 | 박스 prompt 로 첫 마스크 |
| 2 | 점 prompt (**TODO 1**) |
| 3 | negative point 로 다듬기 (**TODO 2**) |
| 4 | 마스크 배열 다루기 — 면적 · 오려내기 · IoU (**TODO 3**) |
| 5 | YOLO + SAM 자동 마스크 생성기 (**TODO 4**) |
| 6 | 내 이미지로 |

**시작 전에** — `런타임 → 런타임 유형 변경 → T4 GPU`. CPU 에서는 한 번 추론에 10초 넘게 걸립니다.

## STEP 0 · 환경 준비

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110


def show(img_bgr, title=None, w=10):
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1])          # BGR -> RGB
    plt.axis("off")
    if title:
        plt.title(title, fontsize=13)
    plt.show()


def overlay(img_bgr, mask, color=(60, 120, 240), alpha=0.55):
    """마스크를 원본 위에 반투명하게 얹는다. mask 는 (H, W) bool 배열."""
    out = img_bgr.copy()
    layer = np.zeros_like(out)
    layer[mask.astype(bool)] = color
    return cv2.addWeighted(out, 1.0, layer, alpha, 0)


print("준비 완료")

## STEP 1 · 박스 prompt 로 첫 마스크

`sam2.1_b.pt` 는 약 155 MB 입니다. 처음 실행할 때 한 번 내려받습니다.

박스를 주면 **그 박스 안의 주된 물체**를 오려 냅니다. 어제 YOLO 가 낸 박스를 그대로 쓸 수
있다는 뜻이기도 합니다 — STEP 5 에서 실제로 해 봅니다.

In [ ]:
from ultralytics import SAM
from ultralytics.utils import ASSETS

sam = SAM("sam2.1_b.pt")
IMG = str(ASSETS / "bus.jpg")

img = cv2.imread(IMG)
print("이미지 크기:", img.shape)          # (1080, 810, 3)

r = sam(IMG, bboxes=[[4, 229, 796, 728]], verbose=False)[0]   # 버스를 감싸는 박스
print("마스크 텐서:", r.masks.data.shape)  # (N, H, W)

show(r.plot(), "박스 prompt 로 만든 마스크")

`r.plot()` 은 마스크에 색을 입혀 그려 줍니다. 직접 그려 보면 마스크의 정체가 더 분명해집니다.

In [ ]:
m = r.masks.data[0].cpu().numpy()          # (H, W) bool
print(m.shape, m.dtype)
print("True 인 픽셀 수:", m.sum())
print("이미지에서 차지하는 비율: %.1f%%" % (100 * m.sum() / m.size))

show(overlay(img, m), "직접 얹은 마스크")

## STEP 2 · 점으로 가리키기

이번에는 박스 없이 **점 하나**만 찍습니다. `points` 에 픽셀 좌표를, `labels` 에 1(포함)을 줍니다.

먼저 어디를 찍을지 정하기 위해 좌표 격자를 그려 봅니다. (그대로 실행)

In [ ]:
grid = img.copy()
h, w = grid.shape[:2]
for x in range(0, w, 100):
    cv2.line(grid, (x, 0), (x, h), (200, 200, 200), 1)
    cv2.putText(grid, str(x), (x + 4, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
for y in range(0, h, 100):
    cv2.line(grid, (0, y), (w, y), (200, 200, 200), 1)
    cv2.putText(grid, str(y), (6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
show(grid, "좌표 격자 — 어디를 찍을지 골라 보세요")

### TODO 1

왼쪽에 서 있는 사람 위의 한 점을 찍어 마스크를 만드세요. 대략 `(150, 650)` 근처입니다.

힌트 — `sam(IMG, points=[x, y], labels=[1], verbose=False)[0]`

In [ ]:
# TODO 1 ── 점 하나로 사람을 오려 내세요
r1 = ...                     # sam(...) 호출

m1 = r1.masks.data[0].cpu().numpy()
print("면적:", m1.sum(), "픽셀")
show(overlay(img, m1, color=(40, 160, 90)), "점 하나로 만든 마스크")

점을 조금만 옮기면 결과가 꽤 달라집니다. 세 군데를 찍어 비교해 보세요. (그대로 실행)

In [ ]:
pts = [[150, 650], [150, 450], [400, 300]]
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, p in zip(axes, pts):
    rr = sam(IMG, points=p, labels=[1], verbose=False)[0]
    mm = rr.masks.data[0].cpu().numpy()
    o = overlay(img, mm)
    cv2.circle(o, tuple(p), 12, (0, 255, 255), -1)
    ax.imshow(o[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"점 {tuple(p)}  ·  {mm.sum():,}px", fontsize=12)
plt.tight_layout(); plt.show()

**관찰해 보세요** — 같은 사람이라도 어디를 찍느냐에 따라 옷만 나오기도 하고,
사람 전체가 나오기도 합니다. SAM 에게 '무엇이 하나의 물체인가' 는 애초에 답이 여럿인 질문입니다.

## STEP 3 · negative point 로 다듬기

포함할 곳은 `1`, 빼고 싶은 곳은 `0` 을 줍니다. 이때 점들이 **한 객체에 대한 지시**라는 것을
알려 주기 위해 리스트를 한 겹 더 감쌉니다.

```python
points=[[[x1, y1], [x2, y2]]]     # 대괄호가 세 겹
labels=[[1, 0]]                   # 대괄호가 두 겹
```

### TODO 2

버스 위의 점(포함)과 버스 앞 사람 위의 점(제외)을 함께 주어, 사람이 빠진 버스 마스크를
만들어 보세요. 버스는 `(400, 400)`, 사람은 `(150, 650)` 근처입니다.

In [ ]:
# TODO 2 ── 포함 점과 제외 점을 함께 주세요
r2 = sam(IMG,
         points=...,          # [[[포함 x, y], [제외 x, y]]]
         labels=...,          # [[1, 0]]
         verbose=False)[0]

m2 = r2.masks.data[0].cpu().numpy()
o = overlay(img, m2)
cv2.circle(o, (400, 400), 12, (0, 255, 0), -1)      # 초록 = 포함
cv2.circle(o, (150, 650), 12, (0, 0, 255), -1)      # 빨강 = 제외
show(o, "포함 점(초록) + 제외 점(빨강)")

## STEP 4 · 마스크는 배열이다

마스크가 넘파이 배열이라는 것을 알면 할 수 있는 일이 많아집니다.

먼저 배경을 지우고 물체만 남겨 봅니다. (그대로 실행)

In [ ]:
cut = img * m[:, :, None]                  # 마스크 밖은 0(검정)
white = img.copy()
white[~m.astype(bool)] = 255               # 마스크 밖을 흰색으로

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, im, t in zip(axes, [cut, white], ["배경을 검게", "배경을 희게"]):
    ax.imshow(im[:, :, ::-1]); ax.axis("off"); ax.set_title(t, fontsize=12)
plt.tight_layout(); plt.show()

### TODO 3 — 두 마스크의 IoU

오전에 배운 mask IoU 를 함수로 짜세요.

```
IoU = 두 마스크가 겹치는 픽셀 수  ÷  둘 중 하나라도 True 인 픽셀 수
```

힌트 — 불리언 배열끼리는 `&` (그리고), `|` (또는) 연산이 됩니다.

In [ ]:
def mask_iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    # TODO 3 ── 세 줄을 채우세요
    inter = ...
    union = ...
    return ...


# 같은 사람을 서로 다른 점으로 두 번 오려 내 비교해 본다
ra = sam(IMG, points=[150, 650], labels=[1], verbose=False)[0]
rb = sam(IMG, points=[150, 550], labels=[1], verbose=False)[0]
ma = ra.masks.data[0].cpu().numpy()
mb = rb.masks.data[0].cpu().numpy()

print("면적 A:", ma.sum(), " 면적 B:", mb.sum())
print("IoU: %.3f" % mask_iou(ma, mb))

값이 거의 1 이 나왔을 것입니다 — 두 점 모두 같은 사람을 가리켰기 때문입니다.
대조군으로 서로 다른 물체를 가리킨 경우도 재 봅니다. (그대로 실행)

In [ ]:
r_bus = sam(IMG, points=[400, 400], labels=[1], verbose=False)[0]
m_bus = r_bus.masks.data[0].cpu().numpy()

pairs = [("사람 A vs 사람 B (점만 다름)", ma, mb),
         ("사람 A vs 버스",              ma, m_bus),
         ("버스 vs 버스 (자기 자신)",     m_bus, m_bus)]
for name, p, q in pairs:
    print(f"{name:<26} IoU = {mask_iou(p, q):.3f}")

IoU 가 1에 가까우면 두 점이 같은 물체를 가리킨 것이고, 낮으면 서로 다른 것을 잡은 것입니다.
**최종 프로젝트 주제 2번이 정확히 이 실험의 확장판입니다** — 정답 마스크와의 IoU 를 재는 것이죠.

윤곽선만 그리고 싶다면 `masks.xy` 를 씁니다. (그대로 실행)

In [ ]:
contour = ra.masks.xy[0].astype(np.int32)     # (M, 2) 좌표 배열
print("윤곽선 점 개수:", len(contour))

o = img.copy()
cv2.polylines(o, [contour], isClosed=True, color=(0, 200, 255), thickness=4)
show(o, "masks.xy 로 그린 윤곽선")

## STEP 5 · YOLO + SAM

**오늘 실습의 핵심입니다.** 어제 만든 탐지 결과를 그대로 SAM 에 넘깁니다.

- YOLO 는 **이름**을 안다 → `person`, `bus`
- SAM 은 **모양**을 안다 → 픽셀 단위 마스크

### TODO 4

아래 두 줄을 채워 탐지 → 분할 파이프라인을 완성하세요.

힌트
- 박스는 `d.boxes.xyxy.cpu().numpy().tolist()` 로 리스트로 꺼냅니다.
- SAM 에는 `sam(IMG, bboxes=박스리스트, verbose=False)[0]` 로 넘깁니다.

In [ ]:
from ultralytics import YOLO

det = YOLO("yolo11n.pt")
d = det(IMG, conf=0.25, verbose=False)[0]

boxes = ...                  # TODO 4 ── 박스를 리스트로 꺼내기
names = [d.names[int(c)] for c in d.boxes.cls]
print("탐지:", names)

sres = ...                   # TODO 4 ── 그 박스를 SAM 의 prompt 로
masks = sres.masks.data.cpu().numpy()

print("마스크 개수:", len(masks), "  박스 개수:", len(boxes))
for name, mk in zip(names, masks):
    print(f"{name:<10} {int(mk.sum()):>8,} px")

이름이 붙은 마스크를 한 장에 모아 그려 봅니다. (그대로 실행)

In [ ]:
PALETTE = [(232, 96, 21), (50, 113, 233), (91, 125, 46),
           (163, 79, 122), (60, 60, 220), (200, 160, 40)]

out = img.copy()
for k, (name, mk) in enumerate(zip(names, masks)):
    out = overlay(out, mk, color=PALETTE[k % len(PALETTE)], alpha=0.5)
    ys, xs = np.where(mk.astype(bool))
    if len(xs):
        cv2.putText(out, name, (int(xs.mean()) - 30, int(ys.mean())),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 3)

show(out, "이름이 붙은 마스크 — YOLO + SAM")

**이것이 최종 프로젝트 주제 1번의 baseline 입니다.** 여기서부터
정답 마스크와 비교하거나, threshold 를 바꿔 가며 실패를 관찰하면 프로젝트가 됩니다.

### 마스크 면적으로 할 수 있는 것 (그대로 실행)

In [ ]:
total = img.shape[0] * img.shape[1]
print("클래스별 화면 점유율")
for name, mk in zip(names, masks):
    print(f"  {name:<10} {100 * mk.sum() / total:5.2f}%")

overlap = 0
for i in range(len(masks)):
    for j in range(i + 1, len(masks)):
        ov = (masks[i].astype(bool) & masks[j].astype(bool)).sum()
        if ov > 0:
            overlap += 1
            print(f"  겹침: {names[i]} ↔ {names[j]}  {ov:,}px")
if overlap == 0:
    print("  겹치는 마스크 없음")

## STEP 6 · 내 이미지로

어제 STEP 5 에서 올린 이미지를 다시 씁니다. 없다면 아래 셀에서 새로 올려도 됩니다.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    my_images = list(uploaded.keys())
except Exception as e:
    print("Colab 이 아닙니다:", e)
    my_images = []
print(my_images)

In [ ]:
my_images = globals().get("my_images", [])
if not my_images:
    my_images = [str(ASSETS / "zidane.jpg")]

path = my_images[0]
im = cv2.imread(path)

d = det(path, conf=0.25, verbose=False)[0]
bx = d.boxes.xyxy.cpu().numpy().tolist()
nm = [d.names[int(c)] for c in d.boxes.cls]

if bx:
    sr = sam(path, bboxes=bx, verbose=False)[0]
    o = im.copy()
    for k, (name, mk) in enumerate(zip(nm, sr.masks.data.cpu().numpy())):
        o = overlay(o, mk, color=PALETTE[k % len(PALETTE)], alpha=0.5)
    show(o, f"{path}  —  {nm}")
else:
    print("탐지된 객체가 없습니다. conf 를 낮춰 보세요.")

### 확인해 볼 것

1. **탐지가 실패하면 마스크도 없습니다.** COCO 에 없는 물체는 이 파이프라인으로 잡히지 않습니다.
   그런 물체는 점을 직접 찍어 SAM 만 써 보세요 — 그건 잘 됩니다.
2. **겹친 물체**는 어떻게 나오나요? 앞뒤로 겹친 사람 두 명을 SAM 이 나눌 수 있나요?
3. **가는 것**(다리, 손가락, 전선)의 경계는 어떤가요?

---

## 정리

1. 점을 어디에 찍느냐에 따라 …
2. negative point 는 … 할 때 유용하다
3. YOLO + SAM 조합에서 실패의 원인은 크게 … 와 … 로 나뉜다

다음 노트북(`4일차_실습_ONNX_Runtime`)은 3차시에 엽니다.